<a href="https://colab.research.google.com/github/emankhalid019/sql-from-scratch/blob/main/Day03_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Theory + Commands
# Union All, Group By, Having, Exists, Any, All,Select Into, Insert Into Select, Case, Null Functions, Stored Procedures, Comments, Operators (Colab Executable - Python + SQLite)

import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

def run(query):
    """Executes a query. Returns a table if it's a SELECT, else a status message."""
    cur.execute(query)
    conn.commit()
    if query.strip().upper().startswith("SELECT"):
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    return "Query executed successfully."


# sample tables used throughout
run("""
CREATE TABLE Employees (
    EmployeeID INTEGER,
    Name TEXT,
    Age INTEGER,
    Salary INTEGER,
    DepartmentID INTEGER,
    Bonus INTEGER
);
""")

run("""
CREATE TABLE Departments (
    DepartmentID INTEGER,
    DepartmentName TEXT,
    Budget INTEGER
);
""")

run("INSERT INTO Employees VALUES (1, 'Ali', 25, 50000, 1, 2000);")
run("INSERT INTO Employees VALUES (2, 'Sara', 30, 60000, 2, NULL);")
run("INSERT INTO Employees VALUES (3, 'Ahmed', 28, 55000, 1, 1500);")
run("INSERT INTO Employees VALUES (4, 'Ayesha', 35, 70000, 3, 3000);")
run("INSERT INTO Employees VALUES (5, 'Zarmeen', 22, 45000, 2, NULL);")

run("INSERT INTO Departments VALUES (1, 'IT', 200000);")
run("INSERT INTO Departments VALUES (2, 'HR', 90000);")
run("INSERT INTO Departments VALUES (3, 'Finance', 150000);")

print(run("SELECT * FROM Employees;"))
print(run("SELECT * FROM Departments;"))








## SQL UNION ALL
# UNION combines results from multiple SELECT statements and removes duplicates. UNION ALL does the same but KEEPS duplicates(and is faster, since it skips the duplicate-check step)
run("CREATE TABLE FormerEmployees (Name TEXT);")
run("INSERT INTO FormerEmployees VALUES ('Bilal');")
run("INSERT INTO FormerEmployees VALUES ('Ali');")  # duplicate, for demo

print(run("""
SELECT Name FROM Employees
UNION
SELECT Name FROM FormerEmployees;
"""))# 'Ali' appears only once

print(run("""
SELECT Name FROM Employees
UNION ALL
SELECT Name FROM FormerEmployees;
"""))# 'Ali' appears twice, since duplicates are kept








## SQL GROUP BY
# GROUP BY groups rows that have the same values in specified columnsinto summary rows. It's almost always used with aggregate functions (COUNT, SUM, AVG, MIN, MAX)
print(run("""
SELECT DepartmentID, COUNT(*) AS EmployeeCount
FROM Employees
GROUP BY DepartmentID;
"""))

print(run("""
SELECT DepartmentID, AVG(Salary) AS AvgSalary
FROM Employees
GROUP BY DepartmentID;
"""))

print(run("""
SELECT DepartmentID, SUM(Salary) AS TotalSalary
FROM Employees
GROUP BY DepartmentID;
"""))







## SQL HAVING
# HAVING filters groups AFTER aggregation (WHERE cannot be used with aggregate functions directly, so HAVING is used instead).
print(run("""
SELECT DepartmentID, COUNT(*) AS EmployeeCount
FROM Employees
GROUP BY DepartmentID
HAVING COUNT(*) > 1;
"""))# Only departments with more than 1 employee are shown

print(run("""
SELECT DepartmentID, AVG(Salary) AS AvgSalary
FROM Employees
GROUP BY DepartmentID
HAVING AVG(Salary) > 50000;
"""))







## SQL EXISTS
# EXISTS checks whether a subquery returns any rows. Returns TRUE if the subquery finds at least one matching row.
print(run("""
SELECT DepartmentName
FROM Departments
WHERE EXISTS (
    SELECT 1 FROM Employees
    WHERE Employees.DepartmentID = Departments.DepartmentID
    AND Employees.Salary > 65000
);
""")) # Shows departments that have at least one employee earning > 65000











## SQL ANY
# ANY compares a value to ANY value returned by a subquery. Returns TRUE if the condition is true for at least one value.
# Note: SQLite doesn't support ANY directly -- it's rewritten here
# using an equivalent subquery with MAX()/MIN() or IN.
print(run("""
SELECT Name, Salary FROM Employees
WHERE Salary > (SELECT MIN(Salary) FROM Employees WHERE DepartmentID = 1);
"""))
# Equivalent to: WHERE Salary > ANY (SELECT Salary FROM Employees WHERE DepartmentID = 1)

# Standard syntax on databases that support ANY (e.g. MySQL, PostgreSQL):
#   SELECT Name, Salary FROM Employees
#   WHERE Salary > ANY (SELECT Salary FROM Employees WHERE DepartmentID = 1);












## SQL ALL
# ALL compares a value to ALL values returned by a subquery. Returns TRUE only if the condition is true for every value.
print(run("""
SELECT Name, Salary FROM Employees
WHERE Salary > (SELECT MAX(Salary) FROM Employees WHERE DepartmentID = 2);
"""))# Equivalent to: WHERE Salary > ALL (SELECT Salary FROM Employees WHERE DepartmentID = 2)

# Standard syntax on databases that support ALL:
#   SELECT Name, Salary FROM Employees
#   WHERE Salary > ALL (SELECT Salary FROM Employees WHERE DepartmentID = 2);







## SQL SELECT INTO
# SELECT INTO copies data from one table into a brand NEW table.
# Note: SELECT INTO is SQL Server syntax. In SQLite/MySQL, the equivalent is CREATE TABLE ... AS SELECT.
run("""
CREATE TABLE HighEarners AS
SELECT Name, Salary FROM Employees WHERE Salary > 55000;
""")
print(run("SELECT * FROM HighEarners;"))
# Standard SQL Server syntax:
#   SELECT Name, Salary INTO HighEarners
#   FROM Employees WHERE Salary > 55000;









## SQL INSERT INTO SELECT
# INSERT INTO SELECT copies data from one table and inserts it into an EXISTING table (unlike SELECT INTO, which creates a new table)
run("CREATE TABLE Archive (Name TEXT, Salary INTEGER);")
run("""
INSERT INTO Archive (Name, Salary)
SELECT Name, Salary FROM Employees WHERE DepartmentID = 1;
""")
print(run("SELECT * FROM Archive;"))






## SQL CASE
# CASE adds if-then-else logic to a query. It goes through conditions and returns a value as soon as the first condition is met.
print(run("""
SELECT Name, Salary,
CASE
    WHEN Salary >= 65000 THEN 'High'
    WHEN Salary >= 50000 THEN 'Medium'
    ELSE 'Low'
END AS SalaryLevel
FROM Employees;
"""))







# SQL NULL FUNCTIONS
# NULL functions provide a fallback value when a column contains NULL.
# SQLite/MySQL use IFNULL() or COALESCE(); SQL Server uses ISNULL();
# Oracle uses NVL(). COALESCE() works across most databases.

print(run("SELECT Name, IFNULL(Bonus, 0) AS Bonus FROM Employees;"))
print(run("SELECT Name, COALESCE(Bonus, 0) AS Bonus FROM Employees;"))

# Using a NULL function inside a calculation (avoids NULL breaking the sum):
print(run("SELECT Name, Salary + IFNULL(Bonus, 0) AS TotalPay FROM Employees;"))






## SQL STORED PROCEDURES
# A stored procedure is prepared SQL code that can be saved and
# re-used with different parameters, instead of writing the same
# query repeatedly.
# Note: SQLite does NOT support stored procedures. Below is the
# standard syntax as used in SQL Server / MySQL for reference.

# SQL Server syntax:
#   CREATE PROCEDURE GetEmployeesByDept @DeptID INT
#   AS
#   BEGIN
#       SELECT * FROM Employees WHERE DepartmentID = @DeptID;
#   END;
#
#   Executing it:
#   EXEC GetEmployeesByDept @DeptID = 1;

# MySQL syntax:
#   DELIMITER //
#   CREATE PROCEDURE GetEmployeesByDept(IN deptId INT)
#   BEGIN
#       SELECT * FROM Employees WHERE DepartmentID = deptId;
#   END //
#   DELIMITER ;
#
#   Executing it:
#   CALL GetEmployeesByDept(1);

print("Stored procedures aren't supported in SQLite -- see the comments above for real syntax.")







## SQL COMMENTS
# Comments are used to explain code and are ignored by SQL itself.

# Single-line comment(-- This is a single-line comment)
# Multi-line comment()  /* This is a  multi-line comment */)
# Example:
# SELECT * FROM Employees;  -- retrieves all employees
# /* The query below filters
#    employees by department */
# SELECT * FROM Employees WHERE DepartmentID = 1;


# Using comments to disable parts of a query (common real use)
# NOTE: everything below (the /* ... */ block and the -- lines) is written as PYTHON comments here (starting with #), purely to SHOW ou the pattern. They are never sent
# to the database, so they can never cause an error -- this is just for illustration.

# A multi-line comment can "turn off" several whole statements at once:
#   /* SELECT * FROM Employees;
#      SELECT * FROM Departments;
#      SELECT * FROM Archive;
#      SELECT * FROM HighEarners; */
#   SELECT * FROM FormerEmployees;

#   -> Only "SELECT * FROM FormerEmployees;" actually runs.
#      The four SELECT statements inside /* ... */ are skipped entirely.

# A single-line comment can "turn off" just PART of one line:
#   SELECT * FROM Employees -- WHERE DepartmentID = 1;

#   -> Everything after -- on that line is ignored, so the WHERE
#      condition never runs. The query becomes simply:
#      SELECT * FROM Employees
#      This returns ALL employees, not just DepartmentID 1 --
#      a common bug when someone forgets they commented out a filter!

print(run("SELECT * FROM Employees -- WHERE DepartmentID = 1;"))
# Demonstrates the same effect: the WHERE clause is commented out, so every employee is returned instead of only DepartmentID = 1.









## SQL OPERATORS

# Arithmetic Operators: +, -, *, /, %
print(run("SELECT Salary, Salary * 0.1 AS TenPercentBonus FROM Employees;"))

# Comparison Operators: =, >, <, >=, <=, <>
print(run("SELECT * FROM Employees WHERE Salary >= 55000;"))

# Logical Operators: AND, OR, NOT, BETWEEN, LIKE, IN, ALL, ANY, EXISTS
print(run("SELECT * FROM Employees WHERE Salary > 50000 AND DepartmentID = 1;"))

# Bitwise operators (&, |, ^) exist in most SQL databases but are
# rarely used in everyday queries -- SQLite supports & and | for
# bitwise AND / OR on integers.
print(run("SELECT EmployeeID, EmployeeID & 1 AS IsOddID FROM Employees;"))

conn.close()




# Keyword             Purpose                                   Example

# UNION ALL           Combine SELECTs, keeps duplicates          SELECT... UNION ALL SELECT...
# GROUP BY            Group rows for aggregate functions         GROUP BY DepartmentID
# HAVING              Filter groups after aggregation            HAVING COUNT(*) > 1
# EXISTS              True if subquery returns any row           WHERE EXISTS (SELECT ...)
# ANY                 True if condition matches any subquery row WHERE Salary > ANY (SELECT ...)
# ALL                 True if condition matches every row        WHERE Salary > ALL (SELECT ...)
# SELECT INTO         Copy data into a NEW table                 SELECT * INTO NewTable FROM OldTable
# INSERT INTO SELECT  Copy data into an EXISTING table            INSERT INTO T SELECT * FROM OldTable
# CASE                If-then-else logic in a query               CASE WHEN ... THEN ... ELSE ... END
# NULL Functions      Replace NULL with a fallback value          IFNULL(col,0) / COALESCE(col,0)
# Stored Procedures   Saved, reusable SQL code                    CREATE PROCEDURE ... / EXEC / CALL
# Comments            Explanatory notes, ignored by SQL           -- comment  or  /* comment */
# Operators           Arithmetic, comparison, logical, bitwise    +,-,*,/  =,>,<  AND,OR,NOT  &,|

   EmployeeID     Name  Age  Salary  DepartmentID   Bonus
0           1      Ali   25   50000             1  2000.0
1           2     Sara   30   60000             2     NaN
2           3    Ahmed   28   55000             1  1500.0
3           4   Ayesha   35   70000             3  3000.0
4           5  Zarmeen   22   45000             2     NaN
   DepartmentID DepartmentName  Budget
0             1             IT  200000
1             2             HR   90000
2             3        Finance  150000
      Name
0    Ahmed
1      Ali
2   Ayesha
3    Bilal
4     Sara
5  Zarmeen
      Name
0      Ali
1     Sara
2    Ahmed
3   Ayesha
4  Zarmeen
5    Bilal
6      Ali
   DepartmentID  EmployeeCount
0             1              2
1             2              2
2             3              1
   DepartmentID  AvgSalary
0             1    52500.0
1             2    52500.0
2             3    70000.0
   DepartmentID  TotalSalary
0             1       105000
1             2       105000
2           

In [ ]:
#Theory + Commands
#SQL TUTORIAL - PART 4: SQL Database - Create DB, Drop DB, Backup DB, Create Table, Drop Table, Alter Table, Constraints, Not Null, Unique



import sqlite3
import pandas as pd
import os

def run(cur, query):
    """Executes a query. Returns a table if it's a SELECT, else a status message."""
    cur.execute(query)
    if query.strip().upper().startswith("SELECT"):
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    return "Query executed successfully."





## SQL CREATE DB
# CREATE DATABASE creates a brand new database.
# Standard syntax (SQL Server, MySQL, PostgreSQL):
#   CREATE DATABASE database1;

# Note: SQLite has no CREATE DATABASE statement -- a database is simply a FILE. Connecting to a file that doesn't exist yet creates it automatically. Below we create a real file-based database so
# the DB actually exists on disk (unlike ':memory:' used earlier).

db_path = "school.db"
if os.path.exists(db_path):
    os.remove(db_path)   # start fresh each time this cell runs

conn = sqlite3.connect(db_path)   # this line IS the "CREATE DATABASE" step
cur = conn.cursor()
print(f"Database created at: {os.path.abspath(db_path)}")







## SQL DROP DB
# DROP DATABASE permanently deletes an entire database, including every table and all data inside it. Use with extreme caution.
# Standard syntax: DROP DATABASE database1;

# In SQLite, since a database is just a file, "dropping" it means closing the connection and deleting the file:

#   conn.close()
#   os.remove("school.db")

# We will NOT run this now, since we need the database for therest of this tutorial -- this is shown for reference only.











## SQL BACKUP DB

# BACKUP DATABASE creates a full copy of a database, usually storedsafely in case the original is lost or corrupted.
# Standard SQL Server syntax:
#   BACKUP DATABASE database1
#   TO DISK = 'D:\backups\database1.bak';

# In SQLite, a backup is simply a copy of the database file.Python's sqlite3 module also provides a built-in backup() method:
backup_conn = sqlite3.connect("school_backup.db")
conn.backup(backup_conn)
backup_conn.close()
print("Backup created at:", os.path.abspath("school_backup.db"))






## SQL CREATE TABLE
# CREATE TABLE creates a new table with specified columns and types.
# Syntax:
#   CREATE TABLE table_name (
#       column1 datatype constraints,
#       column2 datatype constraints,
#       ...
#   );

print(run(cur, """
CREATE TABLE Departments (
    DepartmentID INTEGER PRIMARY KEY,
    DepartmentName TEXT NOT NULL UNIQUE
);
"""))

print(run(cur, """
CREATE TABLE Employees (
    EmployeeID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Age INTEGER,
    Email TEXT UNIQUE,
    Salary INTEGER,
    DepartmentID INTEGER,
    FOREIGN KEY (DepartmentID) REFERENCES Departments(DepartmentID)
);
"""))

run(cur, "INSERT INTO Departments VALUES (1, 'IT');")
run(cur, "INSERT INTO Departments VALUES (2, 'HR');")
run(cur, "INSERT INTO Employees VALUES (1, 'Ali', 25, 'ali@gmail.com', 50000, 1);")
run(cur, "INSERT INTO Employees VALUES (2, 'Sara', 30, 'sara@gmail.com', 60000, 2);")
conn.commit()

print(run(cur, "SELECT * FROM Departments;"))
print(run(cur, "SELECT * FROM Employees;"))








## SQL DROP TABLE
# DROP TABLE deletes a table completely -- its structure AND all its data. This cannot be undone.
# Syntax:DROP TABLE table_name;

run(cur, "CREATE TABLE TempTable (ID INTEGER, Note TEXT);")
run(cur, "INSERT INTO TempTable VALUES (1, 'temporary data');")
print(run(cur, "SELECT * FROM TempTable;"))

run(cur, "DROP TABLE TempTable;")
conn.commit()
print("TempTable has been dropped -- it no longer exists.")

# TRUNCATE TABLE (not supported in SQLite) empties a table but keeps
# its structure, similar to DELETE FROM table_name; but usually faster
# on large tables in databases that support it (MySQL, SQL Server):
#   TRUNCATE TABLE table_name;











## SQL ALTER TABLE(ALTER TABLE adds, deletes, or modifies columns in an existing table)

# Add a new column
run(cur, "ALTER TABLE Employees ADD COLUMN City TEXT;")
print(run(cur, "SELECT * FROM Employees;"))

# Update the new column with data
run(cur, "UPDATE Employees SET City = 'Lahore' WHERE EmployeeID = 1;")
run(cur, "UPDATE Employees SET City = 'Karachi' WHERE EmployeeID = 2;")
conn.commit()
print(run(cur, "SELECT * FROM Employees;"))

# Rename a column (supported in modern SQLite, MySQL, SQL Server)
run(cur, "ALTER TABLE Employees RENAME COLUMN City TO Location;")
conn.commit()
print(run(cur, "SELECT * FROM Employees;"))

# Standard syntax for dropping a column (SQLite 3.35+, MySQL, SQL Server):
#   ALTER TABLE Employees DROP COLUMN Location;
# Standard syntax for changing a column's data type (MySQL/SQL Server):
#   ALTER TABLE Employees ALTER COLUMN Salary BIGINT;   -- SQL Server
#   ALTER TABLE Employees MODIFY COLUMN Salary BIGINT;  -- MySQL
# Note: SQLite has very limited support for changing column types directly.






## SQL CONSTRAINTS
# Constraints are rules enforced on columns to maintain data accuracy
# and integrity. Common constraints:
#   NOT NULL    -> column cannot store NULL
#   UNIQUE      -> all values in the column must be different
#   PRIMARY KEY -> uniquely identifies each row (NOT NULL + UNIQUE)
#   FOREIGN KEY -> links to a PRIMARY KEY in another table
#   CHECK       -> ensures values meet a specific condition
#   DEFAULT     -> sets a default value when none is provided

print(run(cur, """
CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY,
    ProductName TEXT NOT NULL,
    Price INTEGER CHECK (Price > 0),
    Stock INTEGER DEFAULT 0
);
"""))

run(cur, "INSERT INTO Products (ProductID, ProductName, Price) VALUES (1, 'Laptop', 90000);")
conn.commit()
print(run(cur, "SELECT * FROM Products;"))
# Stock automatically became 0 because of the DEFAULT constraint

# This next insert would FAIL because of the CHECK constraint (Price > 0):
#   INSERT INTO Products (ProductID, ProductName, Price) VALUES (2, 'Faulty Item', -50);
try:
    run(cur, "INSERT INTO Products (ProductID, ProductName, Price) VALUES (2, 'Faulty Item', -50);")
    conn.commit()
except Exception as e:
    print("CHECK constraint blocked this insert:", e)








## SQL NOT NULL
# NOT NULL enforces that a column CANNOT contain a NULL value --
# it must always be given a value when inserting a row.

print(run(cur, """
CREATE TABLE Students (
    StudentID INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Age INTEGER
);
"""))

run(cur, "INSERT INTO Students VALUES (1, 'Ahmed', 20);")
conn.commit()
print(run(cur, "SELECT * FROM Students;"))

# This insert would FAIL because Name is NOT NULL and no value is given:
try:
    run(cur, "INSERT INTO Students (StudentID, Age) VALUES (2, 21);")
    conn.commit()
except Exception as e:
    print("NOT NULL constraint blocked this insert:", e)






##SQL UNIQUE
# UNIQUE ensures all values in a column (or combination of columns)are different -- no duplicates allowed. Unlike PRIMARY KEY, a table can have multiple UNIQUE columns.

run(cur, "INSERT INTO Employees (EmployeeID, Name, Email) VALUES (3, 'Ahmed', 'ahmed@gmail.com');")
conn.commit()
print(run(cur, "SELECT * FROM Employees;"))

# This insert would FAIL because 'ali@gmail.com' already exists (Email is UNIQUE):
try:
    run(cur, "INSERT INTO Employees (EmployeeID, Name, Email) VALUES (4, 'Duplicate', 'ali@gmail.com');")
    conn.commit()
except Exception as e:
    print("UNIQUE constraint blocked this insert:", e)

conn.close()


# Keyword           Purpose                                    Example

# CREATE DATABASE   Creates a new database                      CREATE DATABASE db1;
# DROP DATABASE     Permanently deletes a database               DROP DATABASE db1;
# BACKUP DATABASE   Creates a copy of a database                 BACKUP DATABASE db1 TO DISK = 'path.bak';
# CREATE TABLE      Creates a new table                          CREATE TABLE T (col type, ...);
# DROP TABLE        Deletes a table completely                   DROP TABLE T;
# ALTER TABLE       Adds/modifies/removes columns                ALTER TABLE T ADD COLUMN col type;
# NOT NULL          Column cannot be empty                       Name TEXT NOT NULL
# UNIQUE            No duplicate values allowed                  Email TEXT UNIQUE
# PRIMARY KEY       Uniquely identifies each row                 ID INTEGER PRIMARY KEY
# FOREIGN KEY       Links to another table's primary key         FOREIGN KEY (DeptID) REFERENCES Dept(ID)
# CHECK             Value must satisfy a condition                Price INTEGER CHECK (Price > 0)
# DEFAULT           Sets a default value                          Stock INTEGER DEFAULT 0


Database created at: /content/school.db
Backup created at: /content/school_backup.db
Query executed successfully.
Query executed successfully.
   DepartmentID DepartmentName
0             1             IT
1             2             HR
   EmployeeID  Name  Age           Email  Salary  DepartmentID
0           1   Ali   25   ali@gmail.com   50000             1
1           2  Sara   30  sara@gmail.com   60000             2
   ID            Note
0   1  temporary data
TempTable has been dropped -- it no longer exists.
   EmployeeID  Name  Age           Email  Salary  DepartmentID  City
0           1   Ali   25   ali@gmail.com   50000             1  None
1           2  Sara   30  sara@gmail.com   60000             2  None
   EmployeeID  Name  Age           Email  Salary  DepartmentID     City
0           1   Ali   25   ali@gmail.com   50000             1   Lahore
1           2  Sara   30  sara@gmail.com   60000             2  Karachi
   EmployeeID  Name  Age           Email  Salary  Departm